# From Self-RAG to Agentic RAG: An Enterprise Case Study

**Objective:** Learn when and why to upgrade from Self-RAG to Agentic RAG

## 1. Business Case: Insurance Claims Processing

**Acme Insurance Corp** processes 50,000+ claims monthly.

### Business Problem (5 Key Points)

1. **High Volume, Complex Queries**: Questions span multiple documents
2. **Time-Critical Decisions**: 3 days spent gathering information
3. **Accuracy Requirements**: $2.3M annual cost from incorrect decisions
4. **Knowledge Fragmentation**: 15,000+ documents across systems
5. **Compliance Pressure**: Audit trails required with source citations

## 2. Setup

In [1]:
# Install packages
# Chat/LLM calls use Groq via its OpenAI-compatible API.
# Groq has no embeddings endpoint, so embeddings run locally via sentence-transformers.
%pip install -q openai langchain-openai langchain-huggingface sentence-transformers langchain-chroma langchain-core chromadb python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load environment
import os
from dotenv import load_dotenv
import getpass
load_dotenv()

# Groq serves an OpenAI-compatible API, so we reuse the OpenAI SDK / langchain-openai
# clients and just point them at Groq's base URL with a Groq API key (gsk_...).
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROQ_CHAT_MODEL = "llama-3.3-70b-versatile"

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API Key:")
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found in environment or .env file")

print(f"API Key loaded: ...{os.getenv('GROQ_API_KEY')[-4:]}")

API Key loaded: ...P5KV


In [3]:
# Sample enterprise documents
INSURANCE_DOCUMENTS = [

    # Defines coverage limits, deductibles, exclusions, and claim deadlines for auto insurance policies
    {
        "id": "policy_auto_001",
        "content": """AUTO INSURANCE POLICY TERMS - COMPREHENSIVE COVERAGE

Coverage includes:
- Collision damage: Up to $50,000 per incident
- Theft protection: Full replacement value
- Natural disaster damage: Covered
- Deductible: $500 standard, $1000 for high-risk drivers

Exclusions:
- Racing or competitive events
- Commercial use without rider
- Intentional damage

Claim filing deadline: 30 days from incident.""",
        "metadata": {"type": "policy", "category": "auto"}
    },

    # Specifies coverage limits, exclusions, and special conditions for home insurance policies
    {
        "id": "policy_home_001",
        "content": """HOME INSURANCE POLICY TERMS - STANDARD COVERAGE

Coverage includes:
- Dwelling coverage: Up to $500,000
- Personal property: Up to $250,000
- Liability protection: $100,000 per occurrence

Water damage conditions:
- Burst pipes: Covered if sudden failure
- Flood damage: NOT covered (requires separate policy)
- Sewer backup: Covered with optional rider

Claim filing deadline: 60 days from discovery.""",
        "metadata": {"type": "policy", "category": "home"}
    },

    # Provides internal operational rules for claim prioritization, approvals, documentation, and escalation
    {
        "id": "guideline_claims_001",
        "content": """CLAIMS PROCESSING GUIDELINES - VERSION 3.2

Priority Processing Rules:
- Claims over $10,000: Require supervisor approval
- Claims with injuries: Process within 48 hours
- Multiple claims same policy: Flag for fraud review

Documentation Requirements:
- Photo evidence for property damage
- Police report for theft claims
- Repair estimates from 2 contractors

Escalation: Analyst -> Senior Analyst -> Claims Manager -> VP Claims""",
        "metadata": {"type": "guideline", "category": "claims"}
    },

    # Captures legally mandated claim handling timelines and consumer protection requirements for California
    {
        "id": "regulation_ca_001",
        "content": """STATE REGULATORY REQUIREMENTS - CALIFORNIA

Claims Processing Timelines:
- Acknowledge claim receipt: Within 15 days
- Accept or deny claim: Within 40 days
- Payment after acceptance: Within 30 days

Consumer Protection:
- Must provide written denial explanation
- Must inform of appeal rights

Penalties: Up to $10,000 per violation""",
        "metadata": {"type": "regulation", "state": "CA"}
    },

    # Records historical claims activity and risk assessment for a specific policyholder (John Smith)
    {
        "id": "claims_history_001",
        "content": """CLAIMS HISTORY - John Smith (Policy #AC-2024-78432)

Claim 1 (2023-03-15): Fender bender, $2,340 paid
Claim 2 (2023-08-22): Windshield replacement, $450 paid
Claim 3 (2024-01-10): Catalytic converter theft, $1,800 paid

Risk Assessment: Medium (3 claims in 12 months)
Premium Adjustment: +15% at renewal
Fraud Indicators: None detected""",
        "metadata": {"type": "claims_history", "policyholder": "John Smith"}
    },

    # Defines step-by-step operational procedures for assessing and processing water damage claims
    {
        "id": "procedure_water_001",
        "content": """WATER DAMAGE CLAIMS - SPECIAL PROCEDURES

Step 1: Determine water source
- Category 1 (Clean): Broken supply lines
- Category 2 (Gray): Appliance overflow
- Category 3 (Black): Sewage, flooding

Step 2: Coverage assessment
- Sudden/accidental: Typically covered
- Gradual damage: NOT covered
- Maintenance issues: NOT covered

Required: Photos, plumber report, moisture readings
Processing time: 14-21 days""",
        "metadata": {"type": "procedure", "category": "water_damage"}
    }
]

print(f"Created {len(INSURANCE_DOCUMENTS)} documents")


Created 6 documents


In [11]:
# Setup vector store
# Groq does not offer an embeddings endpoint, so we embed locally with a small
# sentence-transformers model (runs on CPU, downloaded once on first use).
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

documents = [
    Document(page_content=doc["content"], metadata=doc["metadata"])
    for doc in INSURANCE_DOCUMENTS
]

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="acme_insurance"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Vector store ready with {len(documents)} documents")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3265.14it/s]


Vector store ready with 6 documents


## 3. Self-RAG Implementation

In [5]:
from langchain_openai import ChatOpenAI

class SelfRAG:
    def __init__(self, retriever):
        self.retriever = retriever
        # langchain-openai's ChatOpenAI talks to Groq's OpenAI-compatible endpoint.
        self.llm = ChatOpenAI(
            model=GROQ_CHAT_MODEL,
            temperature=0,
            base_url=GROQ_BASE_URL,
            api_key=os.environ["GROQ_API_KEY"],
        )

        self.relevance_prompt = ChatPromptTemplate.from_template("""
Is this document relevant to the query? Answer only RELEVANT or NOT_RELEVANT.

Query: {query}
Document: {document}
""")

        self.generation_prompt = ChatPromptTemplate.from_template("""
You are a claims analyst. Answer using ONLY the provided context.

Context:
{context}

Query: {query}

Answer with specific references to sources.
""")

    def query(self, query: str) -> dict:
        print(f"\n{'='*60}")
        print(f"SELF-RAG: {query[:50]}...")
        print(f"{'='*60}")

        # Retrieve
        print("\n1. Retrieving...")
        docs = self.retriever.invoke(query)
        print(f"   Found {len(docs)} documents")

        # Filter relevant
        print("\n2. Checking relevance...")
        relevant_docs = []
        for i, doc in enumerate(docs):
            chain = self.relevance_prompt | self.llm | StrOutputParser()
            result = chain.invoke({"query": query, "document": doc.page_content[:500]})
            is_relevant = "RELEVANT" in result.upper() and "NOT" not in result.upper()
            print(f"   Doc {i+1}: {'RELEVANT' if is_relevant else 'NOT RELEVANT'}")
            if is_relevant:
                relevant_docs.append(doc)

        if not relevant_docs:
            return {"answer": "No relevant information found.", "sources": []}

        # Generate
        print(f"\n3. Generating from {len(relevant_docs)} docs...")
        context = "\n---\n".join([d.page_content for d in relevant_docs])
        chain = self.generation_prompt | self.llm | StrOutputParser()
        answer = chain.invoke({"context": context, "query": query})

        return {
            "answer": answer,
            "sources": [d.metadata for d in relevant_docs]
        }

self_rag = SelfRAG(retriever)
print("Self-RAG ready!")

Self-RAG ready!


## 4. Self-RAG: Success Scenario

In [6]:
# Simple query - Self-RAG succeeds
simple_query = "What is the deductible for auto insurance?"

result = self_rag.query(simple_query)

print("\n" + "="*60)
print("ANSWER:")
print("="*60)
print(result["answer"])
print(f"\nSources: {result['sources']}")


SELF-RAG: What is the deductible for auto insurance?...

1. Retrieving...
   Found 3 documents

2. Checking relevance...
   Doc 1: RELEVANT
   Doc 2: NOT RELEVANT
   Doc 3: NOT RELEVANT

3. Generating from 1 docs...

ANSWER:
According to the AUTO INSURANCE POLICY TERMS - COMPREHENSIVE COVERAGE, the deductible is $500 standard, and $1000 for high-risk drivers, as stated under the "Deductible" section.

Sources: [{'type': 'policy', 'category': 'auto'}]


## 5. Self-RAG: Failure Scenario

In [7]:
# Complex query - Self-RAG struggles
complex_query = """
A California policyholder John Smith filed a $15,000 water damage claim
from a burst pipe. What approvals are needed and what are the regulatory deadlines?
"""

result = self_rag.query(complex_query)

print("\n" + "="*60)
print("ANSWER:")
print("="*60)
print(result["answer"])
print(f"\nSources: {result['sources']}")
print("\n** Note: Self-RAG may miss some info as it only retrieves once **")


SELF-RAG: 
A California policyholder John Smith filed a $15,...

1. Retrieving...
   Found 3 documents

2. Checking relevance...
   Doc 1: RELEVANT
   Doc 2: RELEVANT
   Doc 3: RELEVANT

3. Generating from 3 docs...

ANSWER:
According to the CLAIMS PROCESSING GUIDELINES - VERSION 3.2, since the claim is over $10,000, it requires supervisor approval (Priority Processing Rules: Claims over $10,000: Require supervisor approval).

As for the regulatory deadlines, the STATE REGULATORY REQUIREMENTS - CALIFORNIA states that we must acknowledge claim receipt within 15 days, accept or deny the claim within 40 days, and make payment after acceptance within 30 days (Claims Processing Timelines). 

Additionally, as this is a water damage claim, we will follow the WATER DAMAGE CLAIMS - SPECIAL PROCEDURES, which requires photos, plumber report, and moisture readings for processing. However, the regulatory deadlines take precedence.

Sources: [{'type': 'procedure', 'category': 'water_damage'}, {'typ

**Issues with the answer**
| # | Decomposed Question Part | Expected from KB | Answer Coverage | Correct? | Issue |
|---|--------------------------|-----------------|-----------------|----------|-------|
| 1 | Water damage source &<br>category | Burst pipe →<br>Category 1 (Clean) | Mentions Category 1<br>(Clean) | ✅ Yes | — |
| 2 | Applicable<br>procedure | Follow Water Damage<br>Special Procedures | Photos, plumber report,<br>moisture readings mentioned | ✅ Yes | — |
| 3 | Claim amount impact<br>($15,000) | Claims > $10,000 <br>require supervisor approval | Not mentioned | ❌ No | Missed mandatory<br>supervisor approval |
| 4 | Claims history<br>consideration | Multiple claims →<br>Flag for fraud review | Not mentioned | ❌ No | Claims history ignored |
| 5 | Fraud indicators | None detected in<br>claims history | Not addressed | ⚠️ Partial | Should state “flagged but<br>no indicators” |
| 6 | CA acknowledgment<br>deadline | Acknowledge claim<br>within 15 days | 15 days stated | ✅ Yes | — |
| 7 | CA accept/deny<br>deadline | Accept or deny within<br>40 days | 40 days stated | ✅ Yes | — |
| 8 | CA payment<br>deadline | Payment within 30 days<br>after acceptance | 30 days stated | ✅ Yes | — |
| 9 | Source attribution | Procedure + Guidelines +<br>Regulation + History | Home policy cited<br>unnecessarily | ❌ No | Noisy / incorrect<br>citation |


## 6. Agentic RAG Architecture

```
User Query --> Orchestrator Agent --> Multiple Tools --> Synthesize Answer
                    |                     |
                    v                     v
              - Plan steps          - Retriever
              - Select tools        - Claims DB lookup
              - Iterate             - Regulation checker
                                    - Deadline calculator
```

| Aspect | Self-RAG | Agentic RAG |
|--------|----------|-------------|
| Retrieval | Single pass | Multiple, dynamic |
| Tools | Just retriever | Multiple specialized |
| Reasoning | Linear | Iterative |

## 7. Agentic RAG Implementation

In [8]:
from datetime import datetime, timedelta
from openai import OpenAI
import json

# Point the OpenAI SDK at Groq's OpenAI-compatible endpoint.
client = OpenAI(base_url=GROQ_BASE_URL, api_key=os.environ["GROQ_API_KEY"])

# Define tools as functions
def retrieve_documents(query: str, doc_type: str = None) -> str:
    """Retrieve documents from knowledge base"""
    results = retriever.invoke(query)
    if doc_type:
        results = [r for r in results if r.metadata.get("type") == doc_type]

    output = []
    for i, doc in enumerate(results[:3]):
        output.append(f"Doc {i+1} ({doc.metadata.get('type', 'unknown')}):")
        output.append(doc.page_content[:500])
    return "\n".join(output) if output else "No documents found"

def lookup_claims_history(name: str) -> str:
    """Look up claims history for a policyholder"""
    for doc in INSURANCE_DOCUMENTS:
        if doc["metadata"].get("type") == "claims_history":
            if name.lower() in doc["content"].lower():
                return doc["content"]
    return f"No claims history found for {name}"

def get_state_regulations(state: str) -> str:
    """Get state regulatory requirements"""
    for doc in INSURANCE_DOCUMENTS:
        if doc["metadata"].get("type") == "regulation":
            if doc["metadata"].get("state", "").upper() == state.upper():
                return doc["content"]
    return f"No regulations found for {state}"

def check_approval_requirements(amount: float) -> str:
    """Check what approvals are needed for claim amount"""
    if amount > 50000:
        return f"${amount:,.0f} claim: VP Claims approval required"
    elif amount > 10000:
        return f"${amount:,.0f} claim: Supervisor approval required"
    else:
        return f"${amount:,.0f} claim: Analyst can approve directly"

def calculate_deadline(days: int, deadline_type: str) -> str:
    """Calculate deadline from today"""
    deadline = datetime.now() + timedelta(days=days)
    return f"{deadline_type}: {deadline.strftime('%Y-%m-%d')} ({days} days from today)"

# Tool definitions for OpenAI
tools = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_documents",
            "description": "Search knowledge base for policy terms, guidelines, procedures",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"},
                    "doc_type": {"type": "string", "description": "Filter by type: policy, guideline, procedure, regulation"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_claims_history",
            "description": "Look up claims history for a policyholder by name",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Policyholder name"}
                },
                "required": ["name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_state_regulations",
            "description": "Get state-specific regulatory requirements and deadlines",
            "parameters": {
                "type": "object",
                "properties": {
                    "state": {"type": "string", "description": "State abbreviation like CA, NY, TX"}
                },
                "required": ["state"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_approval_requirements",
            "description": "Check what approvals are needed based on claim dollar amount",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Claim amount in dollars"}
                },
                "required": ["amount"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_deadline",
            "description": "Calculate a deadline date from today",
            "parameters": {
                "type": "object",
                "properties": {
                    "days": {"type": "integer", "description": "Number of days until deadline"},
                    "deadline_type": {"type": "string", "description": "Type of deadline"}
                },
                "required": ["days", "deadline_type"]
            }
        }
    }
]

# Function to execute tools
def execute_tool(name: str, args: dict) -> str:
    if name == "retrieve_documents":
        return retrieve_documents(args.get("query", ""), args.get("doc_type"))
    elif name == "lookup_claims_history":
        return lookup_claims_history(args.get("name", ""))
    elif name == "get_state_regulations":
        return get_state_regulations(args.get("state", ""))
    elif name == "check_approval_requirements":
        return check_approval_requirements(args.get("amount", 0))
    elif name == "calculate_deadline":
        return calculate_deadline(args.get("days", 0), args.get("deadline_type", ""))
    return "Unknown tool"

print("Agentic RAG tools ready!")

Agentic RAG tools ready!


In [9]:
def agentic_rag_query(query: str, verbose: bool = True) -> dict:
    """Run agentic RAG with tool calling"""

    if verbose:
        print(f"\n{'='*60}")
        print("AGENTIC RAG")
        print(f"{'='*60}")
        print(f"Query: {query[:80]}...")

    messages = [
        {
            "role": "system",
            "content": """You are an expert Claims Analyst Assistant.
Use the available tools to gather all necessary information before answering.
For complex queries, use multiple tools to get complete information.
Always cite your sources."""
        },
        {"role": "user", "content": query}
    ]

    tool_calls_made = []
    max_iterations = 10

    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model=GROQ_CHAT_MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        message = response.choices[0].message

        # If no tool calls, we have the final answer
        if not message.tool_calls:
            if verbose:
                print(f"\n--- Final answer after {len(tool_calls_made)} tool calls ---")
            return {
                "answer": message.content,
                "tool_calls": tool_calls_made
            }

        # Process tool calls
        messages.append(message)

        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)

            if verbose:
                print(f"\n-> Tool: {func_name}")
                print(f"   Args: {func_args}")

            result = execute_tool(func_name, func_args)

            if verbose:
                print(f"   Result: {result[:100]}..." if len(result) > 100 else f"   Result: {result}")

            tool_calls_made.append({"tool": func_name, "args": func_args})

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

    return {"answer": "Max iterations reached", "tool_calls": tool_calls_made}

print("Agentic RAG query function ready!")

Agentic RAG query function ready!


## 8. Agentic RAG: Solving the Complex Query

In [12]:
# Run the complex query with Agentic RAG
complex_query = """
A California policyholder John Smith filed a $15,000 water damage claim
from a burst pipe. Given his claims history, what is the processing procedure,
what approvals are needed, and what are the regulatory deadlines?
"""

result = agentic_rag_query(complex_query)


AGENTIC RAG
Query: 
A California policyholder John Smith filed a $15,000 water damage claim
from a ...

-> Tool: retrieve_documents
   Args: {'doc_type': 'procedure', 'query': 'water damage claim processing procedure'}
   Result: Doc 1 (procedure):
WATER DAMAGE CLAIMS - SPECIAL PROCEDURES

Step 1: Determine water source
- Catego...

-> Tool: lookup_claims_history
   Args: {'name': 'John Smith'}
   Result: CLAIMS HISTORY - John Smith (Policy #AC-2024-78432)

Claim 1 (2023-03-15): Fender bender, $2,340 pai...

-> Tool: check_approval_requirements
   Args: {'amount': 15000}
   Result: $15,000 claim: Supervisor approval required

-> Tool: get_state_regulations
   Args: {'state': 'CA'}
   Result: STATE REGULATORY REQUIREMENTS - CALIFORNIA

Claims Processing Timelines:
- Acknowledge claim receipt...

-> Tool: calculate_deadline
   Args: {'days': 30, 'deadline_type': 'claim processing'}
   Result: claim processing: 2026-07-13 (30 days from today)

--- Final answer after 5 tool calls ---


In [13]:
# Display final answer
print("\n" + "="*60)
print("FINAL ANSWER")
print("="*60)
print(result["answer"])


FINAL ANSWER
For John Smith's $15,000 water damage claim, the processing procedure involves determining the water source and assessing coverage according to the procedure outlined in the document titled "WATER DAMAGE CLAIMS - SPECIAL PROCEDURES". The required documents include photos, a plumber report, and moisture readings. 

Given his claims history, which includes three claims in the past 12 months, his risk assessment is medium, and his premium may be adjusted by +15% at renewal. There are no fraud indicators detected.

For a $15,000 claim, supervisor approval is required, as indicated by the check_approval_requirements function.

The state regulatory requirements for California dictate that the claim receipt must be acknowledged within 15 days, the claim must be accepted or denied within 40 days, and payment must be made within 30 days after acceptance. The consumer protection regulations require a written denial explanation and information about appeal rights.

The deadline for 

In [14]:
# Show tool calls made
print("\n" + "="*60)
print("TOOLS USED")
print("="*60)
for i, tc in enumerate(result["tool_calls"], 1):
    print(f"{i}. {tc['tool']}: {tc['args']}")


TOOLS USED
1. retrieve_documents: {'doc_type': 'procedure', 'query': 'water damage claim processing procedure'}
2. lookup_claims_history: {'name': 'John Smith'}
3. check_approval_requirements: {'amount': 15000}
4. get_state_regulations: {'state': 'CA'}
5. calculate_deadline: {'days': 30, 'deadline_type': 'claim processing'}


## 9. Comparison Summary

| Metric | Self-RAG | Agentic RAG |
|--------|----------|-------------|
| Simple Query | Success | Success |
| Complex Query | Incomplete | Comprehensive |
| Latency | ~2-3 sec | ~8-15 sec |
| Cost | Lower | Higher |
| Tool Calls | 1 | 5-8 |

### When to Use Each:

**Self-RAG**: Simple queries, cost-sensitive, low latency needed

**Agentic RAG**: Complex multi-step queries, compliance requirements, comprehensive answers needed

In [ ]:
# Setup
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Documents
INSURANCE_DOCUMENTS = [
    {"id": "policy_auto", "content": """AUTO INSURANCE POLICY - Deductible: $500 standard, $1000 high-risk. Exclusions: Racing, commercial use, intentional damage. Claim deadline: 30 days.""", "metadata": {"type": "policy", "category": "auto"}},
    {"id": "policy_home", "content": """HOME INSURANCE POLICY - Dwelling: $500,000, Personal property: $250,000. Water damage: Burst pipes covered if sudden. Flood NOT covered. Claim deadline: 60 days.""", "metadata": {"type": "policy", "category": "home"}},
    {"id": "guidelines", "content": """CLAIMS GUIDELINES - Claims over $10,000: Supervisor approval required. Claims with injuries: 48 hour processing. Multiple claims: Fraud review. Escalation: Analyst -> Senior -> Manager -> VP.""", "metadata": {"type": "guideline"}},
    {"id": "ca_regulations", "content": """CALIFORNIA REGULATIONS - Acknowledge receipt: 15 days. Accept/deny: 40 days. Payment: 30 days after acceptance. Penalties: $10,000 per violation.""", "metadata": {"type": "regulation", "state": "CA"}},
    {"id": "john_smith", "content": """CLAIMS HISTORY - John Smith (Policy #AC-2024-78432): 3 claims in 12 months ($2,340 + $450 + $1,800). Risk: Medium. Premium adjustment: +15%. Fraud indicators: None.""", "metadata": {"type": "claims_history", "policyholder": "John Smith"}},
    {"id": "water_procedure", "content": """WATER DAMAGE PROCEDURES - Step 1: Determine source (Clean/Gray/Black). Step 2: Coverage check (Sudden=covered, Gradual=NOT). Step 3: Docs needed (photos, plumber report, moisture readings). Processing: 14-21 days.""", "metadata": {"type": "procedure", "category": "water_damage"}}
]

# Groq has no embeddings endpoint, so embed locally with sentence-transformers.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
documents = [Document(page_content=doc["content"], metadata=doc["metadata"]) for doc in INSURANCE_DOCUMENTS]
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings, collection_name="comparison_test")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Setup complete!")